# Retrieval Augmented Generation

Prompt + Data = Big Success

### Working Environment 

[![Open In Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/build-on-aws/generative-ai-prompt-engineering/blob/main/prompt-engineering-chatbot/prompt-engineering-chatbot.ipynb)


This notebook has been designed, written and tested to run on machines with a minimum of 16GB of RAM (32GB preferred). However, if you don't have access to one sign up for a free account on [Amazon SageMaker Studio Lab](https://studiolab.sagemaker.aws/).  Studio Lab is a free machine learning (ML) development environment that provides compute and storage (up to 15GB) at no cost with NO credit card required.

You can sign up for Amazon SageMaker Studio Lab here: [https://studiolab.sagemaker.aws/]

# Code Example

### Boring Stuff
This is just code needed to set everything up

In [ ]:
%pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
%pip install langchain
%pip install chromadb

In [ ]:
!git lfs install
!git clone https://huggingface.co/johnr9412/Nashville-Meta-Llama-3-8B-Instruct-GGUF ../models

In [ ]:
#MODEL_PATH = "../models/Meta-Llama-3-8B-Instruct-Q4_K_M.gguf"

MODEL_PATH = "/Users/john.robinson/Projects/models/Qwen3-30B-A3B-Instruct-2507-GGUF/Qwen3-30B-A3B-Instruct-2507-Q5_K_M.gguf"

In [ ]:
from llama_cpp import Llama

LLM = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=200, #leave this off unless you have gpu to run against
    verbose=False,
    n_ctx=8000
)

In [ ]:
# Simple string template instead of LangChain PromptTemplate
context_template = """
Given the context below, answer the question that follows. If you do not know the answer and the context does not contain the information to answer the question say you don't know and why.

Context: {context}
Question: {question}

"""

def query_model(user_prompt, additional_context=""):
    messages = [
          {"role": "system", "content": "You are an AI assistant which gives helpful, detailed, and polite answers to the user's questions."},
          {
              "role": "user",
              "content": context_template.format(question=user_prompt, context=additional_context)
          }
      ]

    results = LLM.create_chat_completion(
        messages
    )
    
    answer = results['choices'][0]['message']['content']
    return f"Question: {user_prompt}\nAnswer: {str(answer).strip()}"

### Query Model
First things first: let's ask the model something it won't know

In [ ]:
question = "Who won the 2024 Super Bowl?"

response = query_model(question)
print(response)

Well that was lame.

### Model + Data
Let's give our model some more data to make it more useful

In [ ]:
question = "Who won the 2024 Super Bowl?"
additional_context = "The Kansas City Chiefs won the 2024 Super Bowl 25 to 22 over the San Fransisco 49ers."

response = query_model(question, additional_context=additional_context)
print(response)

# RAG Example

### Store Data
Let's take a document wtih some data detailing who won recent Super Bowls.

Below is some boilerplate to store our data into a local Vector DB.

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB and embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="documents")

# Load documents from kb-docs directory
texts = []
data_dir = "../kb-docs/"

for filename in os.listdir(data_dir):
    file_path = os.path.join(data_dir, filename)
    if os.path.isfile(file_path) and filename.endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            # Simple chunking - split into 1000 character chunks
            for i in range(0, len(content), 1000):
                chunk = content[i:i + 1000].strip()
                if chunk:
                    texts.append(chunk)

print(f"Loaded {len(texts)} text chunks")

# Create embeddings and store in ChromaDB
embeddings = embedding_model.encode(texts)
collection.add(
    ids=[str(i) for i in range(len(texts))],
    embeddings=embeddings.tolist(),
    metadatas=[{"content": text} for text in texts]
)

# Similarity search
question = "Who won the 2024 Super Bowl?"
query_embedding = embedding_model.encode([question])
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=1
)

# Print the most relevant chunk
print("Most relevant content:")
print(results['metadatas'][0][0]['content'])

## THROW IT ALL TOGETHER. 
### Automate the "Retrival" and "Augmate" the "Generation"

In [ ]:
question = "Who won the 2024 Super Bowl?"

query_embedding = embedding_model.encode([question])
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=1
)
additional_context = results['metadatas'][0][0]['content']

response = query_model(question, additional_context=additional_context)
print(response)